## Evaluation V2 - Weighted Combinations (adjusted)

In [1]:
#imports

import pandas as pd
import numpy as np
import random
from collections import Counter
from ingredient_normalisation import normalise_user_ingredients
from evaluation_queries import make_random_queries, make_manual_queries, MIN_SOURCE_INGREDIENTS
from baseline_retrieval import tfidf_model, jaccard_model, minilm_model, recipes_dataset_normalised

# random seeds
split_seed = 34
validation_seed = 83

# evaluation settings
# retrieve top 100 candidates before reranking
candidate_k = 100
# compare recommendations at top 5 and top 10
evaluation_k = [5, 10]
# query settings
ingredients_per_query = 3
query_lengths = [3, 4, 5, 6]
validation_queries = 50

# check number of recipes and columns loaded
print("dataset shape:", recipes_dataset_normalised.shape)

c:\Users\preen\Documents\AIML339 - FINAL\AIML339 - FINAL\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


dataset shape: (10000, 6)


In [2]:
# CREATE VALIDATION AND TEST SET

# select recipes with at least 7 different ingredients
valid_recipes = recipes_dataset_normalised[recipes_dataset_normalised["processing_ingredients"].apply(lambda ingredients: len(set(ingredients)) >= MIN_SOURCE_INGREDIENTS)]

# shuffle the source recipe indices using chosen seed 34
recipe_indices = valid_recipes.index.tolist()
random.Random(split_seed).shuffle(recipe_indices)

# split  recipes into validation and test groups
split_point = int(len(recipe_indices) * 0.5)
validation_indices = recipe_indices[:split_point]
test_indices = recipe_indices[split_point:]



In [3]:
# GENERATE VALIDATION AND MANUAL QUERIES

# generate the same 50 validation source recipes with 3, 4, 5 and 6 ingredients
# use fixed seeds 34 and 83 - queries are generated again each time
validation_query_set = make_random_queries(recipes_dataset_normalised, partition="validation", query_lengths=query_lengths, number_of_queries=validation_queries, split_seed=split_seed, query_seed=validation_seed, min_source_ingredients=MIN_SOURCE_INGREDIENTS)

# create the 20 fixed manual queries - 5 groups with 3, 4, 5 and 6 ingredients
manual_query_set = make_manual_queries()


In [4]:
# EVALUATION - RELEVANCE

#assign relevance score based on matched & missing ingredients
def recipe_relevance(user_ingredients, recipe_ingredients):

    # convert ingredients from user and recipe to sets
    # convert ingredients from user and recipe to sets
    user_set = set(user_ingredients)
    recipe_set = set(recipe_ingredients)

    # calculate matched and missing ingredients
    matched = len(user_set & recipe_set)
    missing = len(recipe_set - user_set)

    # no ingredients matched
    if matched == 0:
        return 0

    # some ingredients matched
    if matched < len(user_set):
        return 1

    # all ingredients matched, with few missing
    if missing <= 2:
        return 3

    # all ingredients matched, but more are missing
    return 2

In [5]:
# CCALCULATE RELEVANCE

# relecance for each recipe using user query
def prepare_query_relevance(user_input):
    #normalise ingredients and convert to set
    user_ingredients = normalise_user_ingredients(user_input)
    # convert ingredients from user and recipe to sets
    user_set = set(user_ingredients)

    # relevance score assigned to each recipe - mark recipes containing all ingredients as relevant
    relevance_scores = recipes_dataset_normalised["processing_ingredients"].apply(lambda ingredients: recipe_relevance(user_ingredients, ingredients))
    binary_relevance = recipes_dataset_normalised["processing_ingredients"].apply(lambda ingredients: user_set.issubset(set(ingredients)))
    # total no. of relevant recipes
    total_relevant = int(binary_relevance.sum())

    return relevance_scores, binary_relevance, total_relevant


In [6]:
#PRECISION, RECALL, NDGC, COVERAGE + MISSING INGREDIENTS

def evaluate_recommendations(recommendations, user_input, k, relevance_scores, binary_relevance, total_relevant):
    # select top k
    top_recipes = recommendations.head(k)
    user_set = set(normalise_user_ingredients(user_input))
    #calculate found relevant in topk
    relevant_found = int(binary_relevance.loc[top_recipes.index].sum())
    # proportion of top k recipes containing all user ingredients
    precision = relevant_found / k
    # proportion of all relevant recipes retrieved - undefined if none are relevant
    recall = relevant_found / total_relevant if total_relevant > 0 else np.nan

    # calculate scores in recommendation orders
    grades = relevance_scores.loc[top_recipes.index].to_numpy()
    #so lower ranked recipes contribute less
    discounts = np.log2(np.arange(2, len(grades) + 2))
    # using relevance scores and rank discounts
    dcg = np.sum((2 ** grades - 1) / discounts)

    #
    # select highest relevance scores in dataset for ideal ranking
    ideal_grades = np.sort(relevance_scores.to_numpy())[::-1][:k]
    ideal_discounts = np.log2(np.arange(2, len(ideal_grades) + 2))
    ideal_dcg = np.sum((2 ** ideal_grades - 1) / ideal_discounts)
    # compare recommendation ranking with ideal ranking
    ndcg = dcg / ideal_dcg if ideal_dcg > 0 else np.nan

    # empty list to add
    coverages = []
    missing_counts = []

    # calculate coverage and missing ingredients for each recommended recipe
    for ingredients in top_recipes["processing_ingredients"]:
        recipe_set = set(ingredients)
        # proportion of user ingredients found in recipe
        coverages.append(len(user_set & recipe_set) / len(user_set))
        # count recipe ingredients user doesnt have
        missing_counts.append(len(recipe_set - user_set))

    return {"precision": precision, "recall": recall, "ndcg": ndcg, "coverage": np.mean(coverages), "missing": np.mean(missing_counts)}


In [7]:
# COMBINED RERANKING V1 AND V2

# calculate how frequently each ingredient occurs in the recipe dataset

ingredient_counts = Counter()

# go through ingredients of each recipe
for ingredients in recipes_dataset_normalised["processing_ingredients"]:
    # count each ingredient once per recipe
    ingredient_counts.update(set(ingredients))

# proportion of recipes containing each ingredient
ingredient_frequencies = {ingredient: count / len(recipes_dataset_normalised) for ingredient, count in ingredient_counts.items()}

# calculate features for each candidate recipe
def prepare_reranking_features(candidates, user_input, similarity_column):
    # candidate recipes copy to store new reranking scores
    reranked = candidates.copy()
    user_set = set(normalise_user_ingredients(user_input))
    # scale similarity scores between 0 and 1
    scores = reranked[similarity_column]
    #lowest and highest
    minimum = scores.min()
    maximum = scores.max()

    # min-max scaling - use 0 if all candidate similarities are equal
    reranked["scaled_similarity"] = 0.0 if maximum == minimum else (scores - minimum) / (maximum - minimum)
    #proportion of user query in recipe
    reranked["coverage"] = reranked["processing_ingredients"].apply(lambda ingredients: len(user_set & set(ingredients)) / len(user_set))
    # proportion of recipe ingredients user doesnt have - original v1 penalty
    reranked["missing_penalty_v1"] = reranked["processing_ingredients"].apply(lambda ingredients: len(set(ingredients) - user_set) / len(set(ingredients)) if ingredients else 0)
    #ingredients user doesnt have
    reranked["missing_count"] = reranked["processing_ingredients"].apply(lambda ingredients: len(set(ingredients) - user_set))
    # reward common missing ingredients
    reranked["common_reward"] = reranked["processing_ingredients"].apply(lambda ingredients: sum(ingredient_frequencies.get(i, 0) for i in set(ingredients) - user_set) / len(set(ingredients)) if ingredients else 0)

    # return candidate recipes with calculated reranking features
    return reranked

# rerank
def rerank_configuration(features, coverage_weight=0.5, missing_weight=0.4, common_weight=0.1, missing_scale=5, missing_method="v2"):
    # copy features so each configuration keeps its own scores
    reranked = features.copy()
    # use original v1 proportion or v2 missing count - cap v2 penalty at 1
    missing_penalty = reranked["missing_penalty_v1"] if missing_method == "v1" else (reranked["missing_count"] / missing_scale).clip(upper=1)
    # calculate final score
    reranked["final_score"] = reranked["scaled_similarity"] + coverage_weight * reranked["coverage"] - missing_weight * missing_penalty + common_weight * reranked["common_reward"]
    # return in order
    # return candidate recipes with calculated reranking features
    return reranked.sort_values("final_score", ascending=False)

In [8]:
# V2 WITH MINIMUM INGREDIENT COVERAGE

# filter recipes that do not match enough user ingredients
# default is 60%, but this can be changed for testing
minimum_coverage = 0.60

def rerank_v2_minimum_coverage(candidates, user_input, similarity_column, coverage_threshold=minimum_coverage):
    # calculate reranking features for the retrieved recipes
    # calculate features once for all coverage thresholds
    features = prepare_reranking_features(candidates, user_input, similarity_column)

    # only keep recipes meeting the minimum coverage
    filtered_features = features[features["coverage"] >= coverage_threshold].copy()

    # rerank the remaining recipes using the original v2 weights
    # apply original v2 weights to recipes remaining after filter
    recommendations = rerank_configuration(filtered_features, coverage_weight=0.5, missing_weight=0.4, common_weight=0.1)

    # return recipes in reranked order after coverage filter
    return recommendations


In [9]:
# RERANKING CONFIGURATIONS - CHANGE ONE ASPECT AT A TIME

# original v1 and v2 weights, test each feature
configurations = {
    "Baseline": None,
    "Combined V1": {"coverage_weight": 0.3, "missing_weight": 0.2, "common_weight": 0.1, "missing_method": "v1"},
    "V1 without common": {"coverage_weight": 0.3, "missing_weight": 0.2, "common_weight": 0.0, "missing_method": "v1"},
    "V2 penalty only": {"coverage_weight": 0.3, "missing_weight": 0.2, "common_weight": 0.1},
    "V2 coverage 0.5": {"coverage_weight": 0.5, "missing_weight": 0.2, "common_weight": 0.1},
    "Combined V2": {"coverage_weight": 0.5, "missing_weight": 0.4, "common_weight": 0.1},
    "V2 without common": {"coverage_weight": 0.5, "missing_weight": 0.4, "common_weight": 0.0},
    "V2 missing 0.6": {"coverage_weight": 0.5, "missing_weight": 0.6, "common_weight": 0.1},
    "V2 missing scale 3": {"coverage_weight": 0.5, "missing_weight": 0.4, "common_weight": 0.1, "missing_scale": 3},
    "V2 missing scale 8": {"coverage_weight": 0.5, "missing_weight": 0.4, "common_weight": 0.1, "missing_scale": 8},
    "Coverage only": {"coverage_weight": 0.5, "missing_weight": 0.0, "common_weight": 0.0},
    "Missing only": {"coverage_weight": 0.0, "missing_weight": 0.4, "common_weight": 0.0},
    "Common only": {"coverage_weight": 0.0, "missing_weight": 0.0, "common_weight": 0.1}
}

# display weights and penalty settings for each configuration
display(pd.DataFrame([{"configuration": name, **(settings or {})} for name, settings in configurations.items()]).fillna("-"))

,configuration,coverage_weight,missing_weight,common_weight,missing_method,missing_scale
0,Baseline,-,-,-,-,-
1,Combined V1,0.3,0.2,0.1,v1,-
2,V1 without common,0.3,0.2,0.0,v1,-
3,V2 penalty only,0.3,0.2,0.1,-,-
4,V2 coverage 0.5,0.5,0.2,0.1,-,-
5,Combined V2,0.5,0.4,0.1,-,-
6,V2 without common,0.5,0.4,0.0,-,-
7,V2 missing 0.6,0.5,0.6,0.1,-,-
8,V2 missing scale 3,0.5,0.4,0.1,-,3.0
9,V2 missing scale 8,0.5,0.4,0.1,-,8.0


In [10]:
%%time
#EVALUATE RANDOM AND MANUAL QUERIES


#store results for each model, map each model to name
# store results for each model, map each model to name
evaluation_results = []
models = {"TF-IDF": (tfidf_model, "tfidf_similarity"), "Jaccard": (jaccard_model, "jaccard_similarity"), "MiniLM": (minilm_model, "minilm_similarity")}

# use the same evaluation for random and manual queries
query_sets = {"random": validation_query_set, "manual": manual_query_set}

# run random and manual queries through same retrieval models
for experiment, query_set in query_sets.items():
    # go through every query
    for query_number, query_row in query_set.iterrows():
        user_input = query_row["query"]
        # calculate relevance across dataset once for this query
        relevance_scores, binary_relevance, total_relevant = prepare_query_relevance(user_input)

        # go through every retrieval model
        for model_name, (model_function, similarity_column) in models.items():
            # retrieve top 100 recipes using original model similarity
            baseline_candidates = model_function(user_input, candidate_k)
            # calculate features once per model and query - reuse for every configuration
            features = prepare_reranking_features(baseline_candidates, user_input, similarity_column)

            # all configurations rerank the same 100 baseline candidates
            # evaluate each configuration
            for configuration_name, settings in configurations.items():
                # keep original ranking for baseline or apply chosen reranking weights
                recommendations = baseline_candidates if settings is None else rerank_configuration(features, **settings)

                # calculate metrics separately for top 5 and top 10
                for k in evaluation_k:
                    metrics = evaluate_recommendations(recommendations, user_input, k, relevance_scores, binary_relevance, total_relevant)
                    evaluation_results.append({"experiment": experiment, "seed": validation_seed if experiment == "random" else None, "query_id": query_row["query_id"], "group": query_row.get("group", "random"), "query_length": query_row["query_length"], "query": user_input, "model": model_name, "configuration": configuration_name, "k": k, "total_relevant": total_relevant, **metrics})


# convert results to df
evaluation_results = pd.DataFrame(evaluation_results)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5743.94it/s]


CPU times: total: 22min 17s
Wall time: 2min 39s


In [11]:
# SUMMARISE RANDOM AND MANUAL EVALUATION

# random query results
# keep random and manual results separate
random_results = evaluation_results[evaluation_results["experiment"] == "random"].copy()
# average metrics for each model, configuration, query length and top k
random_summary = random_results.groupby(["model", "configuration", "query_length", "k"])[["precision", "recall", "ndcg", "coverage", "missing"]].mean().round(4).reset_index()

# manual query results
manual_results = evaluation_results[evaluation_results["experiment"] == "manual"].copy()

# results for each fixed manual group
manual_summary = manual_results.groupby(["group", "model", "configuration", "query_length", "k"])[["precision", "recall", "ndcg", "coverage", "missing"]].mean().round(4).reset_index()


In [12]:
# COMPARE CONFIGURATIONS AT K=5

# compare configurations separately for each query length
results_k5 = random_results[random_results["k"] == 5]
# average top 5 metrics for each model, configuration and query length
comparison_k5 = results_k5.groupby(["model", "configuration", "query_length"])[["precision", "recall", "ndcg", "coverage", "missing"]].mean().round(4)

display(comparison_k5)


precision  recall    ndcg  coverage  \
model   configuration     query_length                                        
Jaccard Baseline          3                 0.304  0.4212  0.7383    0.7160   
                          4                 0.292  0.7012  0.8633    0.7240   
                          5                 0.260  0.8900  0.9495    0.7024   
                          6                 0.252  0.9230  0.9809    0.6813   
        Combined V1       3                 0.348  0.4687  0.7662    0.7360   
...                                           ...     ...     ...       ...   
TF-IDF  V2 penalty only   6                 0.232  0.9213  0.9609    0.6307   
        V2 without common 3                 0.396  0.5988  0.8335    0.7320   
                          4                 0.308  0.7521  0.8787    0.6970   
                          5                 0.244  0.8814  0.9332    0.6560   
                          6                 0.236  0.9280  0.9661    0.6220   

                                        missing  
model   configuration     query_length           
Jaccard Baseline          3               2.748  
                          4               3.284  
                          5               3.412  
                          6               3.308  
        Combined V1       3               2.984  
...                                         ...  
TF-IDF  V2 penalty only   6               3.452  
        V2 without common 3               3.840  
                          4               3.608  
                          5               3.380  
                          6               3.096  

[156 rows x 5 columns]

In [13]:
# DISPLAY ABLATION RESULTS FOR COMMON INGREDIENTS

# select configurations with and without common ingredient reward for comparison
common_comparison = comparison_k5.loc[pd.IndexSlice[:, ["Combined V1", "V1 without common", "Combined V2", "V2 without common", "Common only"]], :]

display(common_comparison)

precision  recall    ndcg  coverage  \
model   configuration     query_length                                        
Jaccard Combined V1       3                 0.348  0.4687  0.7662    0.7360   
                          4                 0.336  0.7448  0.8935    0.7480   
                          5                 0.272  0.9015  0.9614    0.7168   
                          6                 0.264  0.9510  0.9883    0.6973   
MiniLM  Combined V1       3                 0.228  0.3684  0.6496    0.6333   
                          4                 0.220  0.5706  0.7625    0.6380   
                          5                 0.204  0.7536  0.8354    0.6168   
                          6                 0.228  0.9082  0.9318    0.6200   
TF-IDF  Combined V1       3                 0.416  0.6040  0.8499    0.7533   
                          4                 0.316  0.7374  0.8788    0.7180   
                          5                 0.244  0.8787  0.9354    0.6744   
                          6                 0.240  0.9320  0.9632    0.6480   
Jaccard V1 without common 3                 0.348  0.4687  0.7662    0.7347   
                          4                 0.336  0.7448  0.8935    0.7480   
                          5                 0.272  0.9015  0.9614    0.7176   
                          6                 0.264  0.9510  0.9883    0.6987   
MiniLM  V1 without common 3                 0.228  0.3684  0.6498    0.6333   
                          4                 0.220  0.5706  0.7625    0.6380   
                          5                 0.204  0.7536  0.8339    0.6144   
                          6                 0.228  0.9082  0.9384    0.6233   
TF-IDF  V1 without common 3                 0.416  0.6040  0.8495    0.7533   
                          4                 0.316  0.7374  0.8811    0.7180   
                          5                 0.244  0.8787  0.9354    0.6752   
                          6                 0.240  0.9320  0.9635    0.6480   
Jaccard Combined V2       3                 0.344  0.4762  0.7634    0.7240   
                          4                 0.304  0.7298  0.8656    0.7190   
                          5                 0.256  0.8986  0.9383    0.6864   
                          6                 0.248  0.9190  0.9790    0.6487   
MiniLM  Combined V2       3                 0.208  0.3442  0.6438    0.6320   
                          4                 0.228  0.6045  0.7745    0.6490   
                          5                 0.204  0.7601  0.8429    0.6224   
                          6                 0.232  0.9083  0.9471    0.6233   
TF-IDF  Combined V2       3                 0.400  0.5989  0.8343    0.7360   
                          4                 0.308  0.7521  0.8777    0.7000   
                          5                 0.244  0.8814  0.9332    0.6576   
                          6                 0.236  0.9280  0.9661    0.6247   
Jaccard V2 without common 3                 0.344  0.4762  0.7634    0.7240   
                          4                 0.304  0.7298  0.8686    0.7170   
                          5                 0.256  0.8986  0.9383    0.6864   
                          6                 0.248  0.9190  0.9790    0.6473   
MiniLM  V2 without common 3                 0.212  0.3470  0.6458    0.6333   
                          4                 0.224  0.6023  0.7728    0.6470   
                          5                 0.204  0.7601  0.8431    0.6224   
                          6                 0.232  0.9083  0.9471    0.6227   
TF-IDF  V2 without common 3                 0.396  0.5988  0.8335    0.7320   
                          4                 0.308  0.7521  0.8787    0.6970   
                          5                 0.244  0.8814  0.9332    0.6560   
                          6                 0.236  0.9280  0.9661    0.6220   
Jaccard Common only       3                 0.320  0.4332  0.7509    0.7293   
           

In [14]:
# SAVE EVALUATION RESULTS

# save random and manual results separately
random_results.to_csv("evaluation2_randomquery.csv", index=False)
random_summary.to_csv("evaluation2_randomquery_summary.csv", index=False)
manual_results.to_csv("evaluation2_manual_userqueries.csv", index=False)
manual_summary.to_csv("evaluation2_manual_userqueries_summary.csv", index=False)

print("random and manual evaluation results saved")


random and manual evaluation results saved


In [15]:
%%time
# V2 MINIMUM COVERAGE VALIDATION

# compare original v2 against 50%, 60% and 70% minimum coverage
# use the same 200 random validation queries from seed 83
coverage_thresholds = {"V2 original": None, "V2 50%": 0.50, "V2 60%": 0.60, "V2 70%": 0.70}
# store metrics and number of recipes returned for each coverage threshold
coverage_results = []

for query_number, query_row in validation_query_set.iterrows():
    user_input = query_row["query"]
    relevance_scores, binary_relevance, total_relevant = prepare_query_relevance(user_input)

    # retrieve the same 100 tfidf candidates for each threshold
    candidates = tfidf_model(user_input, candidate_k)
    # calculate features once for all coverage thresholds
    features = prepare_reranking_features(candidates, user_input, "tfidf_similarity")

    # compare each coverage threshold using same candidate recipes
    for configuration_name, threshold in coverage_thresholds.items():
        # original v2 has no coverage filter
        filtered_features = features if threshold is None else features[features["coverage"] >= threshold].copy()
        # apply original v2 weights to recipes remaining after filter
        recommendations = rerank_configuration(filtered_features, coverage_weight=0.5, missing_weight=0.4, common_weight=0.1)

        # only evaluate the top 5 recipes
        top_recipes = recommendations.head(5)
        metrics = evaluate_recommendations(recommendations, user_input, 5, relevance_scores, binary_relevance, total_relevant)

        # how many recommendations meet the filter and how many need at most 2 extras
        number_returned = len(top_recipes)
        # proportion of returned recipes needing at most 2 extra ingredients
        at_most_two_missing = (top_recipes["missing_count"] <= 2).mean() if number_returned > 0 else np.nan

        # save query metrics, threshold and whether five recipes were returned
        coverage_results.append({"seed": validation_seed, "query_id": query_row["query_id"], "query_length": query_row["query_length"], "query": user_input, "configuration": configuration_name, "minimum_coverage": threshold, "k": 5, "number_returned": number_returned, "five_returned": int(number_returned == 5), "at_most_two_missing": at_most_two_missing, **metrics})

# convert coverage results to df
coverage_results = pd.DataFrame(coverage_results)

# summarise each threshold and query length
coverage_summary = coverage_results.groupby(["configuration", "query_length"], sort=False).agg(
    queries=("query_id", "count"),
    precision=("precision", "mean"),
    recall=("recall", "mean"),
    ndcg=("ndcg", "mean"),
    coverage=("coverage", "mean"),
    missing=("missing", "mean"),
    average_returned=("number_returned", "mean"),
    five_returned=("five_returned", "mean"),
    at_most_two_missing=("at_most_two_missing", "mean")
).round(4).reset_index()

# save only the detailed results and summary
coverage_results.to_csv("evaluation2_coverage_validation.csv", index=False)
coverage_summary.to_csv("evaluation2_coverage_validation_summary.csv", index=False)

# display average metrics and number returned for each threshold and query length
display(coverage_summary)


,configuration,query_length,queries,precision,recall,ndcg,coverage,missing,average_returned,five_returned,at_most_two_missing
0,V2 original,3,50,0.400,0.5989,0.8343,0.7360,3.8880,5.00,1.00,0.3640
1,V2 50%,3,50,0.412,0.6074,0.8270,0.8162,4.6100,4.70,0.90,0.2360
2,V2 60%,3,50,0.412,0.6074,0.8270,0.8162,4.6100,4.70,0.90,0.2360
3,V2 70%,3,50,0.608,0.7359,0.8220,1.0000,6.0570,3.04,0.40,0.0860
4,V2 original,4,50,0.308,0.7521,0.8777,0.7000,3.6400,5.00,1.00,0.3720
5,V2 50%,4,50,0.308,0.7521,0.8715,0.7483,4.1973,4.84,0.94,0.2620
6,V2 60%,4,50,0.336,0.7708,0.8318,0.8633,5.1853,4.24,0.72,0.1680
7,V2 70%,4,50,0.336,0.7708,0.8318,0.8633,5.1853,4.24,0.72,0.1680
8,V2 original,5,50,0.244,0.8814,0.9332,0.6576,3.3960,5.00,1.00,0.4200
9,V2 50%,5,50,0.244,0.8814,0.8977,0.7805,4.5340,4.48,0.82,0.2640


CPU times: total: 6.56 s
Wall time: 7.02 s
